In [3]:
import pandas as pd
import numpy as np

In [4]:
class ID3:
    def __init__(self):
        self.tree = None

    def entropy(self, y):
        probs = y.value_counts(normalize=True)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def information_gain(self, data, feature, target):
        total_entropy = self.entropy(data[target])
        values = data[feature].unique()

        weighted_entropy = 0
        for val in values:
            subset = data[data[feature] == val]
            weighted_entropy += (len(subset) / len(data)) * self.entropy(subset[target])

        return total_entropy - weighted_entropy

    def build_tree(self, data, features, target):
        y = data[target]

        # Если все объекты одного класса
        if len(y.unique()) <= 1:
            return y.iloc[0]

        # Если признаков больше нет — возвращаем самый частый класс
        if not features:
            return y.value_counts().idxmax()

        # Выбираем лучший признак по Information Gain
        gains = {f: self.information_gain(data, f, target) for f in features}
        best_feature = max(gains, key=gains.get)

        tree = {best_feature: {}}
        remaining_features = [f for f in features if f != best_feature]

        # Рекурсивно строим ветви для каждого значения признака
        for val in data[best_feature].unique():
            subset = data[data[best_feature] == val]
            tree[best_feature][val] = self.build_tree(subset, remaining_features, target)

        return tree

    def fit(self, X, y):
        data = X.copy()
        target_name = 'target'
        data[target_name] = y
        self.tree = self.build_tree(data, list(X.columns), target_name)

    def predict_one(self, sample, tree):
        if not isinstance(tree, dict):
            return tree
        feature = list(tree.keys())[0]
        value = sample[feature]
        if value not in tree[feature]:
            return None # Значение не встречалось при обучении
        return self.predict_one(sample, tree[feature][value])

    def predict(self, X):
        return X.apply(lambda row: self.predict_one(row, self.tree), axis=1)


In [5]:
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)

In [6]:
# Дискретизация: превращаем числа в 3 категории (bins)
for col in df.columns:
    df[col] = pd.cut(df[col], bins=3, labels=['low', 'medium', 'high'])

y = pd.Series(iris.target)

In [7]:
# Обучение
model = ID3()
model.fit(df, y)

In [8]:
import pprint

# Результат
print("Построенное дерево:")
pprint.pprint(model.tree)

Построенное дерево:
{'petal width (cm)': {'high': {'petal length (cm)': {'high': np.int64(2),
                                                     'medium': {'sepal width (cm)': {'low': np.int64(2),
                                                                                     'medium': {'sepal length (cm)': {'medium': np.int64(2)}}}}}},
                      'low': np.int64(0),
                      'medium': {'petal length (cm)': {'high': {'sepal length (cm)': {'high': np.int64(2),
                                                                                      'medium': {'sepal width (cm)': {'low': np.int64(2),
                                                                                                                      'medium': np.int64(1)}}}},
                                                       'medium': {'sepal length (cm)': {'high': np.int64(1),
                                                                                        'low': {'sepal width (cm)

In [9]:
# Проверка на одном примере
test_sample = df.iloc[0]
prediction = model.predict_one(test_sample, model.tree)
print(f"\nПредсказание для первого примера: {prediction} (Реальный класс: {y[0]})")


Предсказание для первого примера: 0 (Реальный класс: 0)


In [11]:
from sklearn.model_selection import train_test_split

# Подготовка данных (используем дискретизированный ирис) и разбиение
X = df
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Инициализация и обучение модели
model_id3 = ID3()
model_id3.fit(X_train, y_train)


In [13]:
# Получение предсказаний для тестовой выборки
predictions = model_id3.predict(X_test)

In [14]:
# Функция для расчета точности (Accuracy)
def calculate_accuracy(y_true, y_pred):
    # Убираем None, если модель встретила неизвестное значение признака
    valid_indices = y_pred.notnull()
    correct = (y_true[valid_indices] == y_pred[valid_indices]).sum()
    return correct / len(y_true)

accuracy = calculate_accuracy(y_test, predictions)

print(f"Точность (Accuracy) на тестовой выборке: {accuracy:.2%}")

# Сравнение первых 10 результатов
results_comparison = pd.DataFrame({'Реальность': y_test, 'Предсказание': predictions})
print("\nСравнение первых 10 ответов:")
print(results_comparison.head(10))

Точность (Accuracy) на тестовой выборке: 100.00%

Сравнение первых 10 ответов:
     Реальность  Предсказание
73            1             1
18            0             0
118           2             2
78            1             1
76            1             1
31            0             0
64            1             1
141           2             2
68            1             1
82            1             1
